# Student Grade Predictor: Early Support Risk Classification

This guided notebook runs the project code and reviews its holdout results. It uses the UCI mathematics records. The project predicts whether final grade `G3` is below 10/20 from selected study-habit and support indicators. Earlier grades `G1` and `G2` are deliberately excluded from the model inputs.

**Learning note:** this is a small educational exercise. The model is not a validated tool for making decisions about students.

## 1. Locate the project files

Run locally from the project folder, or open this notebook in Colab after uploading the repository to GitHub. If the script is not found, the cell asks for the HTTPS URL of your public repository and clones it into the Colab runtime.

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'student_grade_predictor.py').exists():
        project_dir = candidate
        break

if project_dir is None:
    repo_url = input('Paste the HTTPS URL of your public student-grade-predictor repository: ').strip()
    project_dir = Path('/content/student-grade-predictor')
    subprocess.run(['git', 'clone', repo_url, str(project_dir)], check=True)

sys.path.insert(0, str(project_dir))
print(f'Project folder: {project_dir}')

## 2. Train the model and compare it with a baseline

The split is stratified and uses seed 42. Accuracy is shown with balanced accuracy, precision, recall, and F1 for the at-risk/fail class.

In [ ]:
import pandas as pd
from student_grade_predictor import run_analysis, print_summary

results = run_analysis(
    data_path=project_dir / 'data' / 'student-mat.csv',
    output_dir=project_dir / 'outputs',
)
print_summary(results)

## 3. Read the confusion matrix

Here, the positive class means `G3 < 10`. A false negative is an at-risk record predicted as pass; a false positive is a pass record predicted as at risk. The counts are from one 79-record test split.

In [ ]:
from IPython.display import display

cm = results['model_metrics']['confusion_matrix']
confusion = pd.DataFrame(
    [[cm['true_negative'], cm['false_positive']],
     [cm['false_negative'], cm['true_positive']]],
    index=['Actual pass / not at risk', 'Actual at risk / fail'],
    columns=['Predicted pass / not at risk', 'Predicted at risk / fail'],
)
display(confusion)

## 4. Inspect the largest model coefficients

These standardized coefficients describe patterns the fitted model used on this split. They are associations, not proof that changing a feature would change a student's grade.

In [ ]:
coefficients = pd.DataFrame(results['largest_standardized_coefficients'])
coefficients['coefficient'] = coefficients['coefficient'].round(3)
display(coefficients)

## 5. What the results mean

The majority baseline can have higher accuracy by predicting the common pass class every time, while missing every at-risk case. The logistic model catches some at-risk records but also makes false alarms and misses others. Scores may change with a different split; this small study should not be used to label or make decisions about individual students.

**Data citation:** Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5TG7T>. CC BY 4.0.